Revenue Prediction
==================
In this project the goal is to predict movies revenue using their features. revenue is given by ( box_office - budget ) so the main goal is to predict box office. 

1. Prepare and preprocess the given data. 

2. After exploring data find, select and especially create new features. ignore others.

3. Prepare features to feed the model. 

4. Select and try different models.

5. Document and report each step using relative plots and a brief explanation. finally report the best suited model and justify why did it performed well.



- Keep in mind that in this task accuracy itself only has only part of score.

- Hint: to create new features you can use credit attributes. Think of it this way, what affects box office?  

**Tools** 

importing useful tools and libraries. you may use any other library as well.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Modelling
from sklearn import preprocessing, svm 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Tree Visualisation
from sklearn.tree import export_graphviz
from IPython.display import Image
import graphviz

In [2]:
np.random.seed(42)

In [24]:
# Your project struct must look like this,

# |── Name_SID.zip
# │   ├── data
# │   │       ├── rotten_tomatoes_5000_credits.csv
# │   │       ├── rotten_tomatoes_5000_movies.csv
# │   ├── *.ipynb
# │   ├── document.pdf

df_credit = pd.read_csv(r"data/rotten_tomatoes_5000_credits.csv")
df_movies = pd.read_csv(r"data/rotten_tomatoes_5000_movies.csv")

In [22]:
df_movies = df_movies[df_movies['rt_box_office']!=0]
y = df_movies['rt_box_office']
X = df_movies.drop(columns='rt_box_office')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25) 

In [25]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   rt_production_budget     4803 non-null   int64  
 1   rt_genres                4803 non-null   object 
 2   rt_website               1712 non-null   object 
 3   rt_movie_id              4803 non-null   int64  
 4   rt_keywords              4803 non-null   object 
 5   rt_original_language     4803 non-null   object 
 6   rt_original_title        4803 non-null   object 
 7   rt_synopsis              4800 non-null   object 
 8   rt_audience_score        4803 non-null   float64
 9   rt_studios               4803 non-null   object 
 10  rt_production_countries  4803 non-null   object 
 11  rt_release_date          4802 non-null   object 
 12  rt_box_office            4803 non-null   int64  
 13  rt_runtime               4801 non-null   float64
 14  rt_languages            

In [26]:
df_credit.head()

,rt_movie_id,rt_title,rt_actors,rt_staff
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [27]:
# Check for missing values in df_movies
missing_values_movies = df_movies.isnull().sum()
print("Missing values in df_movies:")
print(missing_values_movies[missing_values_movies > 0])

Missing values in df_movies:
rt_website         3091
rt_synopsis           3
rt_release_date       1
rt_runtime            2
rt_tagline          844
dtype: int64


In [28]:
# Check for missing values in df_credit
missing_values_credit = df_credit.isnull().sum()
print("\nMissing values in df_credit:")
print(missing_values_credit[missing_values_credit > 0])



Missing values in df_credit:
Series([], dtype: int64)


اینجا صرفا میخواستم ببینم گه حذف ستون سایت درسته یا نه 
اومدم چک کردم دیدم میانگین فروش اونایی که سایت دارن بیشتر از اونایی که سایت ندارن درنتیجه نگهش میداریم
و Null valueهاش رو با Unkown پر میکنیم 
با No Website یهتره

In [29]:
df_movies['has_website'] = df_movies['rt_website'].notnull().astype(int)
print(df_movies.groupby('has_website')['rt_box_office'].mean())

has_website
0    5.051492e+07
1    1.395772e+08
Name: rt_box_office, dtype: float64


In [30]:
df_movies.loc[:, 'rt_website'] = df_movies['rt_website'].fillna('No Website')

چک کردم ببینم چه مقادیری داخلش داره که اون 3 تا Null value اش رو هم با متن مثلا پر کنم 

In [31]:
# Display unique values for each column with missing values
print("Unique values in rt_synopsis:")
print(df_movies['rt_synopsis'].unique()[:10])  # Show first 10 unique values

Unique values in rt_synopsis:
['In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'
 'Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.'
 'A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE.'
 "Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation and is subsequently hunted by the Gotham City Police Department. Eight years later, Batman encounters the mysterious Selina Kyle and the villainous Bane, a new terrorist leader who overwhelms Gotham's finest. The Dark Kni

In [32]:
df_movies.loc[:, 'rt_synopsis'] = df_movies['rt_synopsis'].fillna('No Synopsis')

در نهایت چون سورت خاصی نداره تاریخ هامون نمیدونم ۱ دونه مقدار missing value ای که داره رو با چی پرش کنم

In [20]:
# Check if 'rt_release_date' is sorted in ascending order
is_sorted = df_movies['rt_release_date'].is_monotonic_increasing
print("Is 'rt_release_date' sorted in ascending order?", is_sorted)

Is 'rt_release_date' sorted in ascending order? False


In [33]:
# Sort the dataframe by 'rt_release_date'
df_movies.sort_values(by='rt_release_date', inplace=True)

# Reset the index after sorting
df_movies.reset_index(drop=True, inplace=True)

# Verify if the column is now sorted
is_sorted = df_movies['rt_release_date'].is_monotonic_increasing
print("Is 'rt_release_date' sorted after sorting?", is_sorted)

Is 'rt_release_date' sorted after sorting? False


In [34]:
# Convert 'rt_release_date' to datetime format
df_movies['rt_release_date'] = pd.to_datetime(df_movies['rt_release_date'], errors='coerce')

# Check for invalid dates
invalid_dates = df_movies['rt_release_date'].isnull().sum()
print(f"Number of invalid dates after conversion: {invalid_dates}")

Number of invalid dates after conversion: 1


In [35]:
df_movies.sort_values(by='rt_release_date', inplace=True)
df_movies.reset_index(drop=True, inplace=True)

# Verify sorting
is_sorted = df_movies['rt_release_date'].is_monotonic_increasing
print("Is 'rt_release_date' sorted after fixing?", is_sorted)

Is 'rt_release_date' sorted after fixing? False


In [36]:
# Check rows with invalid dates (NaT)
invalid_date_rows = df_movies[df_movies['rt_release_date'].isnull()]
print(invalid_date_rows)

      rt_production_budget rt_genres  rt_website  rt_movie_id rt_keywords  \
4802                     0        []  No Website       380097          []   

     rt_original_language           rt_original_title  \
4802                   en  America Is Still the Place   

                                            rt_synopsis  rt_audience_score  \
4802  1971 post civil rights San Francisco seemed li...                0.0   

     rt_studios  ... rt_release_date rt_box_office  rt_runtime  rt_languages  \
4802         []  ...             NaT             0         0.0            []   

     rt_release_status rt_tagline                    rt_title  \
4802          Released        NaN  America Is Still the Place   

     rt_critics_score  rt_review_count  has_website  
4802              0.0                0            0  

[1 rows x 21 columns]


In [37]:
# Display unique values or sample values
print(df_movies['rt_runtime'].unique()[:10])  # First 10 unique values
print(df_movies['rt_runtime'].head(10))  # First 10 values

[197. 151. 153. 109. 100. 127.  89.  66. 105. 101.]
0    197.0
1    151.0
2    153.0
3    109.0
4    100.0
5    127.0
6     89.0
7     89.0
8     66.0
9    105.0
Name: rt_runtime, dtype: float64


مقدار عددی زمان اجرای فیلم و با میانه پر کردم که robust باشه در برابر outlier ها

In [39]:
# Fill missing values in 'rt_runtime' with the median
df_movies.loc[:, 'rt_runtime'] = df_movies['rt_runtime'].fillna(df_movies['rt_runtime'].median())